In [1]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
import urllib.request
import cv2
import numpy as np
import pandas as pd
import os
from datetime import datetime
from keras.models import load_model

# Load Face Detector and Model

classifier = cv2.CascadeClassifier(
    r"C:\Users\suman\OneDrive\Desktop\face_recognition\project_face_detect\haarcascade_frontalface_default.xml"
)

model = load_model(
    r"C:\Users\suman\OneDrive\Desktop\face_recognition\project_face_detect\final_model.h5"
)

URL = 'http://10.30.251.10:8080/shot.jpg'

attendance_file = "Attendance.xlsx"

labels = ['Archita', 'Baijayanti', 'Suman']

# To prevent multiple entries while person stays in front
last_detected = {}
cooldown = 15   # seconds


# Prediction Label

def get_pred_label(pred):
    return labels[pred]


# Image Preprocessing
def preprocess(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (100, 100))
    img = cv2.equalizeHist(img)
    img = img.reshape(1,100,100,1)
    img = img / 255
    return img


# Create Excel File

if not os.path.exists(attendance_file):
    df = pd.DataFrame(columns=["Date","Name","Entry Time","Exit Time"])
    df.to_excel(attendance_file,index=False)


# Attendance Function

def mark_attendance(name):

    now = datetime.now()

    date = now.strftime("%d-%m-%Y")
    time = now.strftime("%H:%M:%S")

    # Cooldown to avoid repeated detection
    if name in last_detected:
        diff = (now - last_detected[name]).total_seconds()
        if diff < cooldown:
            return

    last_detected[name] = now

    df = pd.read_excel(attendance_file)

    today = df[(df["Date"]==date) & (df["Name"]==name)]

    if len(today)==0:

        new_row = {
            "Date":date,
            "Name":name,
            "Entry Time":time,
            "Exit Time":""
        }

        df = pd.concat([df,pd.DataFrame([new_row])],ignore_index=True)

        print(name,"Entry Recorded")

    else:

        index = today.index[-1]

        if pd.isna(df.loc[index,"Exit Time"]) or df.loc[index,"Exit Time"]=="":

            df.loc[index,"Exit Time"]=time

            print(name,"Exit Recorded")

    df.to_excel(attendance_file,index=False)


# Camera Loop

while True:

    img_url = urllib.request.urlopen(URL)

    image = np.array(bytearray(img_url.read()),np.uint8)

    frame = cv2.imdecode(image,-1)

    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    faces = classifier.detectMultiScale(gray,1.3,5)

    for (x,y,w,h) in faces:

        face = frame[y:y+h,x:x+w]

        prediction = model.predict(preprocess(face),verbose=0)

        person = get_pred_label(np.argmax(prediction))

        mark_attendance(person)

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

        cv2.putText(frame,
                    person,
                    (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255,0,0),
                    2)

    cv2.imshow("Attendance System",frame)

    if cv2.waitKey(1) & 0xFF==ord('q'):
        break

cv2.destroyAllWindows()

Archita Entry Recorded
Suman Entry Recorded


C:\Users\suman\AppData\Local\Temp\ipykernel_18988\1620113130.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12:12:19' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index,"Exit Time"]=time


Archita Exit Recorded
Baijayanti Entry Recorded
Suman Exit Recorded


In [3]:
import urllib
import cv2
import numpy as np
from keras.models import load_model

classifier = cv2.CascadeClassifier(r"C:\Users\suman\OneDrive\Desktop\face_recognition\project_face_detect\haarcascade_frontalface_default.xml")

model = load_model(r"C:\Users\suman\OneDrive\Desktop\face_recognition\project_face_detect\final_model.h5")

URL = 'http://10.30.251.10:8080/shot.jpg'

def get_pred_label(pred):
    labels = ['Archita', 'Baijayanti', 'Suman']
    return labels[pred]

def preprocess(img):
    img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img,(100,100))
    img = cv2.equalizeHist(img)
    img = img.reshape(1,100,100,1)
    img = img/255
    return img
    


ret = True
while ret:
    
    img_url = urllib.request.urlopen(URL)
    image = np.array(bytearray(img_url.read()),np.uint8)
    frame = cv2.imdecode(image,-1)
    
    faces = classifier.detectMultiScale(frame,1.5,5)
      
    for x,y,w,h in faces:
        face = frame[y:y+h,x:x+w]
        cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),5)
        cv2.putText(frame,get_pred_label(np.argmax(model.predict(preprocess(face)))),
                    (200,200),cv2.FONT_HERSHEY_COMPLEX,1,
                    (255,0,0),2)
        
    cv2.imshow("capture",frame)
    if cv2.waitKey(1)==ord('q'):
        break

cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━

ConnectionAbortedError: [WinError 10053] An established connection was aborted by the software in your host machine